# Bill summary voice

The four things on an electricity bill, said out loud in the language the bill payer
speaks.

Pipeline:

1. pull six fields off the bill with `doc_ai.extract` (needs an API key)
2. hold back every field the extractor was not sure about (no key)
3. turn the numbers and dates into words (no key)
4. translate the words with `text.translate` (needs a key)
5. speak them with `text_to_speech.convert` (needs a key)

Sections 1 to 4 run with no API key at all. Sections 5 to 7 need one.

**Every cell here ships with empty output, and sections 5 to 7 have never been run.**
There was no Sarvam API key on the machine this recipe was written on. Nothing below is
a recorded result, and no number in it was measured against the live API.

**No bill ships with this recipe.** Put your own at `sample_data/your-bill.pdf` for
sections 5 to 7. Sections 1 to 4 work on an extraction result we made up.

In [ ]:
%pip install -q sarvamai python-dotenv

## 1. The four things, and the rule about them

Four numbers on the bill decide the month: how much is owed, by when, how many units
were used, and what it costs if the money is late. This recipe reads those, plus the
consumer number so the listener knows whose bill it is, plus the disconnection date.

The rule that shapes everything else: **a field is spoken only if the extractor said it
was sure, and only if the value can actually be read.** Anything else is named as
missing rather than guessed at. A wrong due date read out with confidence is worse than
no due date.

The parsing, the gate and the sentence building live in `bill_voice.py` next to this
notebook. It imports nothing outside the Python standard library and never touches the
network, so everything in sections 1 to 4 runs offline.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from bill_voice import (
    BILL_SCHEMA,
    DEFAULT_CONFIDENCE_THRESHOLD,
    EXAMPLE_PAYLOAD,
    MAX_SUMMARY_CHARS,
    bill_schema_json,
    compose_summary,
    parse_indian_amount,
    parse_indian_date,
    say_date,
    say_digits,
    say_rupees,
    say_units,
    select_bill_fields,
)

print("summary budget    :", MAX_SUMMARY_CHARS, "characters")
print("confidence gate at:", DEFAULT_CONFIDENCE_THRESHOLD)

## 2. Reading what the bill actually prints

A bill prints `1,23,456.50`. Python cannot read that, and the two obvious workarounds
are both wrong:

```python
float("1,23,456.50")            # ValueError
"1,23,4567".replace(",", "")    # 1234567 - a typo turned into a confident wrong number
```

So the parser checks the grouping before it trusts the digits. Indian grouping and
western grouping are both accepted; a mixture of the two is refused. The result is a
`decimal.Decimal`, never a float - one in every eighteen two-decimal amounts loses a
paisa under `int(float(s) * 100)`.

Dates are read **day first**, because that is how Indian utility bills print them. The
raw string and the spoken reading are printed side by side below so a wrong reading is
visible instead of silent. A two-digit year is refused: `05/09/25` could be 1925 or 2025.

In [ ]:
for printed in ["1,23,456.50", "123456.50", "Rs. 1,23,456.50", "INR 1,23,456.50",
                "\u20b91,23,456.50", "1,234,567.00", "0.00"]:
    print(f"{printed:>18}  ->  {parse_indian_amount(printed)!r}")

print()
for broken in ["1,23,4567", "12,3456", "1234.567", "-1,234.50", "see reverse", "N/A"]:
    try:
        parse_indian_amount(broken)
    except ValueError as error:
        print(f"{broken:>18}  ->  refused: {str(error).split('. ')[0]}")

print()
for printed in ["05/09/2025", "05-09-2025", "22/09/2025"]:
    read = parse_indian_date(printed)
    print(f"{printed:>18}  ->  {read}  ->  {say_date(read)}")

print()
for broken in ["25/13/2025", "31/02/2025", "00/09/2025", "05/09/25", "next month"]:
    try:
        parse_indian_date(broken)
    except ValueError:
        print(f"{broken:>18}  ->  refused")

## 3. Turning the numbers into words

Nothing with a digit in it reaches the translate call. Amounts are said on the Indian
scale, so `1,23,456` is one lakh twenty three thousand, not one hundred twenty three
thousand - lakh and crore are the units the listener counts in. The scale stops at
crore; above ninety nine crore the renderer refuses rather than reaching for arab.

The consumer number is said one digit at a time on purpose. It is an identifier, not a
quantity. Read as money a ten-digit consumer number is either meaningless or refused,
and neither is a reading of an identifier.

The word `rupees` is spelled out and the rupee sign never reaches the speech call. That
is not a linter rule. It is that nobody here has sent that character to `bulbul:v3` and
we have no way to find out from this machine what it does with it.

In [ ]:
for printed in ["1,23,456.50", "1,234.50", "12345678.00", "1.00", "0.01", "0.00",
                "99,99,99,999.99"]:
    print(f"{printed:>18}  ->  {say_rupees(parse_indian_amount(printed))}")

print()
for printed in ["286", "1", "1,024.5"]:
    print(f"{printed:>18}  ->  {say_units(parse_indian_amount(printed))}")

print()
print("consumer number, spoken as an identifier:")
print("  ", say_digits("9876543210"))
print("the same digits handed to the money renderer:")
try:
    say_rupees(parse_indian_amount("9876543210"))
except ValueError as error:
    print("   refused:", str(error).split('. ')[0])

## 4. The schema, the gate, and the summary

`BILL_SCHEMA` is six flat fields, every one of them declared `string`. That is
deliberate: the bill prints `1,23,456.50`, and asking the extractor for a `number`
throws away the grouping evidence and hands back a float, which is exactly what section
2 exists to avoid. We want the characters as printed.

The schema goes to the API as a **JSON string**, which is what the `schema` parameter is
typed as. Handing it the Python dict fails deep in the HTTP layer with a message that
names neither the parameter nor the problem, so `bill_schema_json()` is the only way out
of this module.

The payload below is a fixture. **It was authored by us, in the shape documented in the
sarvamai 0.1.30 `doc_ai.extract` docstring - "annotations mirroring the result shape
where every leaf has confidence and sources" - and was never captured from a live
response.** Its consumer number, its amounts and its dates are invented. Note that
`annotations` is typed `Dict[str, Any]` in the SDK, so nothing pins that shape but the
prose in that docstring; if the live shape differs, the gate is where you adjust.

The `disconnection_notice` leaf carries a confidence of 0.61, below the 0.80 threshold,
so the summary below drops it and says one item was left out.

### Related work in this repository

This recipe embeds its own six-field schema and its own short confidence check. A fuller
version of both - a schema linter, a four-schema pack and a general gate - is in PR #168
(`examples/doc-extraction-schemas`). This one is deliberately minimal so that it works
whether or not #168 is merged. If both land, the duplication is a few dozen lines and we
would be happy to consolidate them into whichever shape the maintainers prefer.

In [ ]:
print("schema fields:")
for name, spec in BILL_SCHEMA["properties"].items():
    print(f"  {name:>21}  {spec['type']}")
print()
print("what the API is given:", type(bill_schema_json()).__name__,
      "of", len(bill_schema_json()), "characters")

print()
print("the fixture, result half - the values as printed on the bill:")
print(json.dumps(EXAMPLE_PAYLOAD["result"], indent=2))

print()
print("one annotations leaf in full, to show the documented shape:")
one_leaf = EXAMPLE_PAYLOAD["annotations"]["amount_due"]
print(json.dumps({"amount_due": one_leaf}, indent=2))

print()
print("the confidence the gate reads from each leaf:")
for name, leaf in EXAMPLE_PAYLOAD["annotations"].items():
    print(f"  {name:>21}  {leaf['confidence']}")

selection = select_bill_fields(EXAMPLE_PAYLOAD)
print()
print("spoken:")
for name, raw in selection.accepted.items():
    print(f"  {name:>21}  {raw}")
print("held back for a human:")
for name, reason in selection.needs_human_check:
    print(f"  {name:>21}  {reason}")

summary = compose_summary(selection)
print()
print(summary)
print()
print("length:", len(summary), "of a budget of", MAX_SUMMARY_CHARS)

## 5. From here on you need an API key

Everything above ran on a fixture. Everything below calls the live API, costs money, and
**has never been run** - there was no key on the machine this was written on, which is
why every cell ships with empty output.

Put your key in a `.env` file next to this notebook, and your own bill at
`sample_data/your-bill.pdf`.

The key is passed to the client explicitly. `SarvamAI` reads its default at import time,
so a `load_dotenv()` that runs after the import is too late and the client comes up with
no key at all.

The extract call is given `classification` and `auto_orient` as the strings `"false"`
and `"true"`, not Python booleans, because the endpoint takes them as text. No `model`
is passed: the rules file in this repository lists no document-extraction model, and
inventing one here would be a guess dressed up as a fact.

In [ ]:
import base64
import os
import time

from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()
if not os.environ.get("SARVAM_API_KEY"):
    raise RuntimeError(
        "No key found. Copy .env.example to .env and put your Sarvam key in it."
    )

client = SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])

BILL_PATH = Path("sample_data/your-bill.pdf")
if not BILL_PATH.exists():
    raise RuntimeError(
        f"No bill at {BILL_PATH}. This recipe ships no bill on purpose; "
        "photograph or download your own and put it there."
    )

job = client.doc_ai.extract(
    file=[(BILL_PATH.name, BILL_PATH.read_bytes())],
    schema=bill_schema_json(),
    classification="false",
    auto_orient="true",
)
print("job started:", job.job_id, job.status)

## 6. Wait for the job, then gate what comes back

Extraction runs as a job. Poll the status until it reaches one of the four terminal
values - `completed`, `partially_completed`, `failed`, `rejected` - and only then fetch
the results.

`partially_completed` is not a failure and is not treated as one here: some fields came
back and the gate will hold back the rest, which is exactly what it is for.

Then the summary is translated. The model is **`mayura:v1`**, and the target has to be
one of the languages that model supports. This matters more than it looks: the SDK's
`target_language_code` **Literal** carries all twenty-three codes regardless of model, so
passing a language `mayura:v1` does not support type-checks perfectly well and then fails
at the server. That check is ours to make, not the type checker's. For a language outside
the mayura list, switch the model to `sarvam-translate:v1`, whose limit is 2000
characters rather than 1000.

The 1000-character budget the composer works to comes from here: the sarvamai 0.1.30
`text.translate` docstring puts `mayura:v1` at 1000 characters, and translate runs before
the speech call. The tightest cap in the chain is the one that governs.

In [ ]:
TERMINAL_STATUSES = {"completed", "partially_completed", "failed", "rejected"}

status = client.doc_ai.get_status(job.job_id)
while status.status not in TERMINAL_STATUSES:
    time.sleep(5)
    status = client.doc_ai.get_status(job.job_id)
print("final status:", status.status)

if status.status in {"failed", "rejected"}:
    raise RuntimeError(f"The extraction job ended as {status.status}; nothing to read.")

results = client.doc_ai.get_results(job.job_id)
payload = {"result": results.result, "annotations": results.annotations}

selection = select_bill_fields(payload)
print("held back for a human:", selection.needs_human_check)

summary = compose_summary(selection)
print(summary)
print("length:", len(summary), "of a budget of", MAX_SUMMARY_CHARS)

In [ ]:
translation = client.text.translate(
    input=summary,
    source_language_code="en-IN",
    target_language_code="hi-IN",
    model="mayura:v1",
    mode="formal",
)
translated_text = translation.translated_text
print(translated_text)
print("translated length:", len(translated_text))

## 7. Say it

`bulbul:v3` takes 2500 characters. The composer guarantees the **English** is under
1000; it cannot guarantee the translation is under 2500, because how much a Hindi or
Telugu rendering grows or shrinks against the English is something we could not measure
without a key. So the length is checked at runtime and the cell raises rather than
guessing.

The parameter is `language_code`. It is not `target_language_code` - that is the
translate parameter, and confusing the two is a bug this repository has already fixed
twice. The model is named explicitly so the choice is visible, and the speaker is taken
from the `bulbul:v3` list: the v2 and v3 speaker lists are not interchangeable.

If you switch the target to Odia, the code is `od-IN`. The rules file in this repository
also lists `or-IN`, which the API rejects; that is issue #157.

In [ ]:
if len(translated_text) > 2500:
    raise RuntimeError(
        f"The translated summary is {len(translated_text)} characters and bulbul:v3 "
        "takes 2500. Shorten the summary, or split it, before speaking it."
    )

speech = client.text_to_speech.convert(
    text=translated_text,
    language_code="hi-IN",
    model="bulbul:v3",
    speaker="shubh",
    output_audio_codec="wav",
)

audio_path = Path("outputs/bill_summary_hi.wav")
audio_path.write_bytes(base64.b64decode(speech.audios[0]))
print("wrote", audio_path, "-", audio_path.stat().st_size, "bytes")

## What to change

- **The language.** Change `target_language_code` in section 6 and `language_code` in
  section 7 together. Both must be a language the chosen models support.
- **The threshold.** `select_bill_fields(payload, threshold=...)`. The default of 0.80 is
  a cautious judgement, not a measurement: nobody here had a key or a stack of bills to
  measure it against. Tune it on your own documents, and watch what lands in
  `needs_human_check` as you do.
- **The fields.** Add one to `BILL_SCHEMA` with a `type` and a real `description`, teach
  `say_field` how to read it, and give it a sentence in `compose_summary`. Keep every
  field a `string`.

## What this does not do

It does not check the bill's arithmetic, recompute a tariff, or offer any advice about
paying or disputing. It reads back four things the bill already says. It makes no claim
about extraction accuracy, because nobody here has measured any.